# Train the DDI classifier (cross-entropy vs Focal Loss) on a free GPU

Runs on **Kaggle** (default) or **Colab**. Needs a GPU session (T4 is enough). Run the cells in order; every run writes its result
files at once, so if the session stops you can start again and finished runs are skipped.

**Before you start**
1. Push the branch `aakaash/webapp` to GitHub (this notebook clones it).
2. Kaggle: create a *private dataset* named `ddi2013` and upload the folder `data/ddi2013/DDICorpus` (11 MB; it holds `Train/` and `Test/`). Add it to this notebook. Colab: put the same folder in Google Drive at `MyDrive/medsafe/ddi2013/DDICorpus`.
3. Turn on the GPU (Kaggle: Settings > Accelerator > GPU T4; Colab: Runtime > Change runtime type).

**Time (an estimate until measured, T4 with mixed precision):** about 5.5 to 8 hours in total. The data are never committed to git.

In [ ]:
!nvidia-smi

In [ ]:
import os, glob, pathlib, subprocess, sys
ON_KAGGLE = os.path.exists('/kaggle/working')
WORK = pathlib.Path('/kaggle/working' if ON_KAGGLE else '/content')
REPO_URL = 'https://github.com/milanjohnson/Medication_Safety_and_Decision_support_system'
BRANCH = 'aakaash/webapp'
if ON_KAGGLE:
    hits = glob.glob('/kaggle/input/*/DDICorpus') + glob.glob('/kaggle/input/*/*/DDICorpus')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    hits = glob.glob('/content/drive/MyDrive/medsafe/ddi2013/DDICorpus')
assert hits, 'DDICorpus folder not found: upload it as described in the first cell'
DATA_SRC = pathlib.Path(hits[0])
print('corpus:', DATA_SRC, sorted(p.name for p in DATA_SRC.iterdir()))

In [ ]:
%cd {WORK}
if not (WORK / 'repo').exists():
    !git clone --depth 1 -b {BRANCH} {REPO_URL} repo
%cd {WORK}/repo/aakaash
!git log --oneline -1
!pip install -q -r requirements-train.txt
import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, torch.cuda.get_device_name(0))

In [ ]:
# Parse the XML into train/dev/test jsonl (nothing from the corpus is committed to git) and check the counts.
import json, shutil
sys.path.insert(0, '.')
from medsafe import ddi2013
stage = WORK / 'ddi_xml' / 'DDICorpus'
if not stage.exists():
    shutil.copytree(DATA_SRC, stage)
stats = ddi2013.build_dataset(stage.parent, pathlib.Path('artifacts/ddi2013'))
n = {s: sum(stats['label_counts'][s].values()) for s in ('train', 'dev', 'test')}
print(n)
assert n == {'train': 25037, 'dev': 2754, 'test': 5715}, 'instance counts differ from the local parser counts'

In [ ]:
# Fail fast: download BioBERT (about 436 MB), build the model, run one forward and backward pass.
!python scripts/check_encoder.py

In [ ]:
# Dev-set search (about 15 short runs). Resumable. The test set is not read here.
!python scripts/train_ddi.py search --amp --out-dir {WORK}/out

In [ ]:
# Final cross-entropy model: up to 20 epochs with early stopping, then the test set once.
if not (WORK / 'out/metrics/ddi_ce.json').exists():
    !python scripts/train_ddi.py final --loss ce --from-search --amp --out-dir {WORK}/out

In [ ]:
# Final focal-loss model, then the faithfulness proxy on 300 dev sentences (Integrated Gradients).
if not (WORK / 'out/metrics/ddi_focal.json').exists():
    !python scripts/train_ddi.py final --loss focal --from-search --amp --faithfulness 300 --out-dir {WORK}/out

In [ ]:
# Side by side, as measured. Read this before you report anything.
for name in ('ce', 'focal'):
    j = json.load(open(WORK / f'out/metrics/ddi_{name}.json'))
    print(name, 'config', j['config'], 'best epoch', j['best_epoch'])
    print('  dev  micro', {k: round(v, 4) for k, v in j['dev']['micro'].items() if k in ('precision', 'recall', 'f1')})
    print('  test micro', {k: round(v, 4) for k, v in j['test']['micro'].items() if k in ('precision', 'recall', 'f1')})

In [ ]:
# Bundle the files to download. Models are large (about 0.45 GB each): the focal model is the one the app uses.
!cd {WORK}/out && ls -la models metrics metrics/ddi_runs
!cd {WORK} && zip -r -q ddi_results.zip out/models/ddi_focal out/metrics && ls -la ddi_results.zip
!cd {WORK} && zip -r -q ddi_ce_model.zip out/models/ddi_ce && ls -la ddi_ce_model.zip

## What to download and where it goes (all folders are gitignored; nothing here is committed)

| Download from this session | Put it in (on your computer) |
|---|---|
| `out/models/ddi_focal/` (4 files, the checkpoint the app loads) | `aakaash/artifacts/models/ddi_focal/` |
| `out/models/ddi_ce/` (optional, the baseline) | `aakaash/artifacts/models/ddi_ce/` |
| `out/metrics/ddi_ce.json`, `ddi_focal.json`, `ddi_search.json` | `aakaash/artifacts/metrics/` |
| `out/metrics/ddi_runs/` (dev and test predictions, training logs, package versions) | `aakaash/artifacts/metrics/ddi_runs/` |

Zip files: `ddi_results.zip` (focal model + all metrics) and `ddi_ce_model.zip`. Kaggle: use Output > download; Colab: Files panel.